## データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのplot

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/22:00:00', '20220902/00:00:00']
pt.timespan('2022-09-01/22:20:00', 1, keyword='hour')

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/2230-2330'

psp.themis.fgm(trange=time_range, probe='a', level='l2')                    # fgh: 128 Hz, fhl: 16 Hz, fgs: 2.74 sec
#psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efw')    # efw: 16448 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp')    # efp: 512 Hz
#psp.themis.efi(trange=time_range, probe='a', level='l2')                    # eff: 8 Hz

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

E512_data_gsm = pt.data_quants['tha_efp_gsm']
B128_data_gsm = pt.data_quants['tha_fgh_gsm']

time_range_T = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E512_data_gsm = E512_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B128_data_gsm = B128_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

print(E512_data_gsm)
print('')
print(B128_data_gsm)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

E512_data_gsm_x = E512_data_gsm.isel(v_dim=0)
E512_data_gsm_y = E512_data_gsm.isel(v_dim=1)
E512_data_gsm_z = E512_data_gsm.isel(v_dim=2)

B128_data_gsm_x = B128_data_gsm.isel(v_dim=0)
B128_data_gsm_y = B128_data_gsm.isel(v_dim=1)
B128_data_gsm_z = B128_data_gsm.isel(v_dim=2)

pt.store_data('E512_gsm_x', data={'x': E512_data_gsm_x.time, 'y': E512_data_gsm_x})
pt.store_data('E512_gsm_y', data={'x': E512_data_gsm_x.time, 'y': E512_data_gsm_x})
pt.store_data('E512_gsm_z', data={'x': E512_data_gsm_x.time, 'y': E512_data_gsm_x})
pt.store_data('B128_gsm_x', data={'x': B128_data_gsm_x.time, 'y': B128_data_gsm_x})
pt.store_data('B128_gsm_y', data={'x': B128_data_gsm_y.time, 'y': B128_data_gsm_y})
pt.store_data('B128_gsm_z', data={'x': B128_data_gsm_z.time, 'y': B128_data_gsm_z})

pt.options('E512_gsm_x', 'ytitle', 'E_x (gsm)')
pt.options('E512_gsm_y', 'ytitle', 'E_y (gsm)')
pt.options('E512_gsm_z', 'ytitle', 'E_z (gsm)')
pt.options(['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z'], 'ysubtitle', '[mV/m]')
pt.options('B128_gsm_x', 'ytitle', 'B_x (gsm)')
pt.options('B128_gsm_y', 'ytitle', 'B_y (gsm)')
pt.options('B128_gsm_z', 'ytitle', 'B_z (gsm)')
pt.options(['B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z'], 'ysubtitle', '[nT]')

#pt.tplot(['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z'])

In [ ]:
import numpy as np
import pytplot as pt
import matplotlib.pyplot as plt

gap_thr = np.timedelta64(10, 's')

# ---------- 1. 電場チャンク ----------------------------------
t_ref_E = pt.data_quants['E512_gsm_x'].time.values
is_new_E = np.concatenate(([True], np.diff(t_ref_E) > gap_thr))
chunk_E  = np.cumsum(is_new_E) - 1

# ---------- 2. 磁場チャンク ----------------------------------
t_ref_B = pt.data_quants['B128_gsm_x'].time.values
is_new_B = np.concatenate(([True], np.diff(t_ref_B) > gap_thr))
chunk_B  = np.cumsum(is_new_B) - 1

# ---------- 3. 分割ループ ------------------------------------
vars_E = ['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z']
vars_B = ['B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']

def split_and_store(var_list, t_ref, chunk_id):
    for v in var_list:
        da   = pt.data_quants[v]
        for k in np.unique(chunk_id):
            sel = chunk_id == k
            if sel.sum() == 0:
                continue
            new = f'{v}_seg{k}'
            pt.store_data(new,
                          data={'x': t_ref[sel], 'y': da.values[sel]})
            unit = '[mV/m]' if v.startswith('e_') else '[nT]'
            pt.options(new,'ytitle',v); pt.options(new,'ysubtitle',unit)

split_and_store(vars_E, t_ref_E, chunk_E)
split_and_store(vars_B, t_ref_B, chunk_B)

# ---------- 4. まとめ変数 ------------------------------------
def make_all(base):
    pt.store_data(f'{base}_all', data=pt.tnames(f'{base}_seg*'))

for base in vars_E + vars_B:
    make_all(base)

# ---------- 5. 描画 ------------------------------------------
pt.options('E512_gsm_x_all', 'ytitle', 'E_x (gsm)')
pt.options('E512_gsm_y_all', 'ytitle', 'E_y (gsm)')
pt.options('E512_gsm_z_all', 'ytitle', 'E_z (gsm)')
pt.options(['E512_gsm_x_all', 'E512_gsm_y_all', 'E512_gsm_z_all'], 'ysubtitle', '[mV/m]')
pt.options('B128_gsm_x_all', 'ytitle', 'B_x (gsm)')
pt.options('B128_gsm_y_all', 'ytitle', 'B_y (gsm)')
pt.options('B128_gsm_z_all', 'ytitle', 'B_z (gsm)')
pt.options(['B128_gsm_x_all', 'B128_gsm_y_all', 'B128_gsm_z_all'], 'ysubtitle', '[nT]')

vars_to_plot = ['E512_gsm_x_all','E512_gsm_y_all','E512_gsm_z_all', 'B128_gsm_x_all','B128_gsm_y_all','B128_gsm_z_all']
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pandas as pd

# ────────────────────────────────────────────────
# 0. セグメント名リストを取得
# ────────────────────────────────────────────────
seg_ids = sorted({v.split('_seg')[-1]                      # {'0','1',...}
                  for v in pt.tnames('E512_gsm_x_seg*')})

# まとめ用リスト（後で overplot するために使う）
ex_all, ey_all, ez_all = [], [], []
bx_all, by_all, bz_all = [], [], []

for sid in seg_ids:
    # TVar 取り出し → pandas DataFrame へ （xarray に変換するため）
    def to_df(var):           # var = 'e_per1_seg0' など
        dq = pt.data_quants[var]
        return pd.DataFrame({'time': dq.time.values,
                             var.split('_seg')[0]: dq.values})

    df_E = (to_df(f'E512_gsm_x_seg{sid}')
            .merge(to_df(f'E512_gsm_y_seg{sid}'), on='time')
            .merge(to_df(f'E512_gsm_z_seg{sid}'),  on='time'))

    df_B = (to_df(f'B128_gsm_x_seg{sid}')
            .merge(to_df(f'B128_gsm_y_seg{sid}'), on='time')
            .merge(to_df(f'B128_gsm_z_seg{sid}'), on='time'))

    # ---- xarray Dataset へ ----
    ds_E = xr.Dataset({k:(('time',), df_E[k].to_numpy(dtype=float))
                       for k in ['E512_gsm_x','E512_gsm_y','E512_gsm_z']},
                      coords={'time': df_E['time'].to_numpy('datetime64[ns]')})

    ds_B = xr.Dataset({k:(('time',), df_B[k].to_numpy(dtype=float))
                       for k in ['B128_gsm_x','B128_gsm_y','B128_gsm_z']},
                      coords={'time': df_B['time'].to_numpy('datetime64[ns]')})

    rename_dict_E = {
        'E512_gsm_x': 'E128_gsm_x',
        'E512_gsm_y': 'E128_gsm_y',
        'E512_gsm_z': 'E128_gsm_z'
    }

    # renameメソッドで変数名を変更 (512 Hz -> 128 Hz)
    ds_E_128 = ds_E.rename(rename_dict_E)

    # ---- E を B の時刻へ線形補間 ----
    ds_E_128  = ds_E_128.interp(time=ds_B.time, method='linear')
    ds_merged = xr.merge([ds_E_128, ds_B])
    ds_merged = ds_merged.dropna(dim='time', how='any', subset=['E128_gsm_x', 'E128_gsm_y', 'E128_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z'])

    # ---- TVar 登録（seg ID を引き継ぐ） ----
    for var in ds_merged.data_vars:
        new_name = f'{var}_i_seg{sid}'
        pt.store_data(new_name,
                      data={'x': ds_merged.time.values,
                            'y': ds_merged[var].values})
        unit = '[mV/m]' if var.startswith('E128_') else '[nT]'
        pt.options(new_name, 'ytitle', var)
        pt.options(new_name, 'ysubtitle', unit)

        # まとめリストへ追加
        if   var == 'E128_gsm_x': ex_all.append(new_name)
        elif var == 'E128_gsm_y': ey_all.append(new_name)
        elif var == 'E128_gsm_z': ez_all.append(new_name)
        elif var == 'B128_gsm_x': bx_all.append(new_name)
        elif var == 'B128_gsm_y': by_all.append(new_name)
        elif var == 'B128_gsm_z': bz_all.append(new_name)

# ────────────────────────────────────────────────
# 2. overplot 用の “まとめ変数” を 6 つ作成
# ────────────────────────────────────────────────
pt.store_data('E128_gsm_x_all_i', data=ex_all)
pt.store_data('E128_gsm_y_all_i', data=ey_all)
pt.store_data('E128_gsm_z_all_i', data=ez_all)
pt.store_data('B128_gsm_x_all_i', data=bx_all)
pt.store_data('B128_gsm_y_all_i', data=by_all)
pt.store_data('B128_gsm_z_all_i', data=bz_all)

pt.options('E128_gsm_x_all_i', 'ytitle', 'E_x (gsm)')
pt.options('E128_gsm_y_all_i', 'ytitle', 'E_y (gsm)')
pt.options('E128_gsm_z_all_i', 'ytitle', 'E_z (gsm)')
pt.options(['E128_gsm_x_all_i', 'E128_gsm_y_all_i', 'E128_gsm_z_all_i'], 'ysubtitle', '[mV/m]')
pt.options('B128_gsm_x_all_i', 'ytitle', 'B_x (gsm)')
pt.options('B128_gsm_y_all_i', 'ytitle', 'B_y (gsm)')
pt.options('B128_gsm_z_all_i', 'ytitle', 'B_z (gsm)')
pt.options(['B128_gsm_x_all_i', 'B128_gsm_y_all_i', 'B128_gsm_z_all_i'], 'ysubtitle', '[nT]')

# ────────────────────────────────────────────────
# 3. プロット（6 パネル、各パネルに全セグメントが重なって表示）
# ────────────────────────────────────────────────

vars_to_plot = ['E128_gsm_x_all_i', 'E128_gsm_y_all_i', 'E128_gsm_z_all_i', 'B128_gsm_x_all_i', 'B128_gsm_y_all_i', 'B128_gsm_z_all_i']
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm_interp.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
B_fgs_gsm = pt.data_quants['tha_fgs_gsm']

B_fgs_gsm = B_fgs_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

ds_B_fgs_gsm = xr.Dataset({
    'Bspin_gsm_x': ('time', B_fgs_gsm.isel(v_dim=0).data),
    'Bspin_gsm_y': ('time', B_fgs_gsm.isel(v_dim=1).data),
    'Bspin_gsm_z': ('time', B_fgs_gsm.isel(v_dim=2).data)
}, coords={'time': B_fgs_gsm.time})

ds_B_fgs_gsm

In [ ]:
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        'tha_fgs_gsm',
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'tha_fgs_gsm.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        'tha_fgs_gsm',
        display=True   # 省略可（デフォルト）
    )

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        'tha_fgh_gsm',
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'tha_fgh_gsm.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        'tha_fgh_gsm',
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# 補助関数：単一セグメントのFAC変換処理をまとめる
# ------------------------------------------------------------
def rotate_segment_to_fac(sid, vec_base_name, matrix_da):
    """
    指定されたセグメントのベクトルデータをFACへ変換し、成分に分割する。
    戻り値: 3つの成分変数名のリスト [comp_x, comp_y, comp_z]
    """
    print(f"  - Rotating vector: {vec_base_name}_i_seg{sid}")
    
    # 1. ベクトルデータを取得
    vec_tvar = f'{vec_base_name}_i_seg{sid}'
    t_vec, d_vec = pt.get_data(vec_tvar)

    # 2. 回転行列をベクトルの時間軸に手動で補間
    matrix_interp = matrix_da.interp(time_mat=t_vec, method="linear").values

    # 3. NumPyのeinsumで手動でベクトル回転
    fac_np = np.einsum('tij,tj->ti', matrix_interp, d_vec)
    
    # 4. 新しいtplot変数として格納し、成分分割
    # 出力変数名を 'Ex_fac_seg{sid}' のようなシンプルな形式にする
    prefix = 'E' if 'E' in vec_base_name else 'B'
    fac_tvar = f'{prefix}_fac_vec_seg{sid}'
    pt.store_data(fac_tvar, data={'x': t_vec, 'y': fac_np})
    
    # 5a. まず、デフォルト名でベクトルを成分に分割する
    # この時、戻り値としてデフォルト名のリストが得られる
    # 例: ['E_fac_vec_seg0_x', 'E_fac_vec_seg0_y', 'E_fac_vec_seg0_z']
    default_component_names = psp.split_vec(fac_tvar) 

    # 5b. 我々が望む新しい名前のリストを作成する
    # 例: ['Ex_fac_seg0', 'Ey_fac_seg0', 'Ez_fac_seg0']
    desired_component_names = [f'{prefix}{c}_fac_seg{sid}' for c in ['x', 'y', 'z']]

    # 5c. デフォルト名の変数を、望む名前に一つずつリネームする
    for i in range(3):
        pt.tplot_rename(default_component_names[i], desired_component_names[i])
    
    # 6. 最終的な（リネーム後の）変数名のリストを返す
    return desired_component_names

# ------------------------------------------------------------
# 0. パラメータ設定
# ------------------------------------------------------------
b_field_lf_tvar = 'tha_fgl_gsm' 
e_field_hf_base = 'E128_gsm'
b_field_hf_base = 'B128_gsm'
seg_ids = sorted({v.split('_seg')[-1] for v in pt.tnames(f'{e_field_hf_base}_x_i_seg*')})

# ------------------------------------------------------------
# 1. 成分データをベクトルに結合し、メタデータを設定
# ------------------------------------------------------------
print("--- ステップ1: 成分データをベクトルに結合し、座標系メタデータを設定 ---")
for sid in seg_ids:
    e_components = [f'{e_field_hf_base}_x_i_seg{sid}', f'{e_field_hf_base}_y_i_seg{sid}', f'{e_field_hf_base}_z_i_seg{sid}']
    e_vec_tvar = f'{e_field_hf_base}_i_seg{sid}'
    pt.join_vec(e_components, newname=e_vec_tvar)
    pt.data_quants[e_vec_tvar].attrs['coordinate_system'] = 'gsm'
    
    b_components = [f'{b_field_hf_base}_x_i_seg{sid}', f'{b_field_hf_base}_y_i_seg{sid}', f'{b_field_hf_base}_z_i_seg{sid}']
    b_vec_tvar = f'{b_field_hf_base}_i_seg{sid}'
    pt.join_vec(b_components, newname=b_vec_tvar)
    pt.data_quants[b_vec_tvar].attrs['coordinate_system'] = 'gsm'
print("ベクトル変数の作成とメタデータの設定が完了しました。")

# ------------------------------------------------------------
# 2. FAC回転行列の作成と準備
# ------------------------------------------------------------
print("\n--- ステップ2: FAC回転行列を作成し、準備 ---")
fac_matrix_tvar = psp.fac_matrix_make(b_field_lf_tvar)

t_mat, d_mat = pt.get_data(fac_matrix_tvar)
if d_mat is not None and d_mat.ndim == 2:
    d_mat = d_mat[np.newaxis, :, :]
matrix_da = xr.DataArray(d_mat, dims=('time_mat', 'row', 'col'), coords={'time_mat': t_mat})
print(f"回転行列をxarray.DataArrayとして準備完了。Shape: {matrix_da.shape}")

# ------------------------------------------------------------
# 3. segment ごとに FAC 変換 (補助関数を利用)
# ------------------------------------------------------------
print("\n--- ステップ3: 各セグメントをFACへ変換 ---")
fac_vars = {'Ex': [], 'Ey': [], 'Ez': [], 'Bx': [], 'By': [], 'Bz': []}

for sid in seg_ids:
    print(f"\n--- Processing segment {sid} ---")
    
    # 補助関数を呼び出して電場と磁場をそれぞれ変換
    e_fac_components = rotate_segment_to_fac(sid, e_field_hf_base, matrix_da)
    b_fac_components = rotate_segment_to_fac(sid, b_field_hf_base, matrix_da)
    
    # 結果をリストに格納
    fac_vars['Ex'].append(e_fac_components[0])
    fac_vars['Ey'].append(e_fac_components[1])
    fac_vars['Ez'].append(e_fac_components[2])
    fac_vars['Bx'].append(b_fac_components[0])
    fac_vars['By'].append(b_fac_components[1])
    fac_vars['Bz'].append(b_fac_components[2])

# ------------------------------------------------------------
# 3. まとめ変数を作って overplot
# ------------------------------------------------------------
pt.store_data('Ex_fac_all', data=fac_vars['Ex'])
pt.store_data('Ey_fac_all', data=fac_vars['Ey'])
pt.store_data('Ez_fac_all', data=fac_vars['Ez'])
pt.store_data('Bx_fac_all', data=fac_vars['Bx'])
pt.store_data('By_fac_all', data=fac_vars['By'])
pt.store_data('Bz_fac_all', data=fac_vars['Bz'])

# オプション設定
pt.options('Ex_fac_all', 'ytitle', 'E_x (fac)')
pt.options('Ey_fac_all', 'ytitle', 'E_y (fac)')
pt.options('Ez_fac_all', 'ytitle', 'E_z (fac)')
pt.options(['Ex_fac_all', 'Ey_fac_all', 'Ez_fac_all'], 'ysubtitle', '[mV/m]')
pt.options('Bx_fac_all', 'ytitle', 'B_x (fac)')
pt.options('By_fac_all', 'ytitle', 'B_y (fac)')
pt.options('Bz_fac_all', 'ytitle', 'B_z (fac)')
pt.options(['Bx_fac_all', 'By_fac_all', 'Bz_fac_all'], 'ysubtitle', '[nT]')


# ------------------------------------------------------------
# 4. プロット
# ------------------------------------------------------------
vars_to_plot = ['Ex_fac_all','Ey_fac_all','Ez_fac_all', 'Bx_fac_all','By_fac_all','Bz_fac_all']


if os.path.isdir(path_base_save_plot):
    # --- フォルダが存在する場合：表示せずに保存 ---
    
    # 1. まず、プロットオブジェクトだけを取得する (save_pngは使わない)
    print("Generating plot objects...")
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,                # Jupyter上での自動表示を抑制
        return_plot_objects=True      # fig, axes を返してもらう
    )
    
    # 2. 取得したオブジェクトを使って、手動でファイルを保存する
    save_path = os.path.join(path_base_save_plot, 'EB_fields_fac.png')
    print(f"Saving plot to: {save_path}")
    fig.savefig(save_path, dpi=200) # dpiもここで指定できる
    
    # 3. メモリを解放する
    plt.close(fig)
    print("Plotting complete and memory released.")

else:
    # --- フォルダが存在しない場合：通常表示 ---
    pt.tplot(
        vars_to_plot,
        display=True
    )

In [ ]:
import pytplot as pt
import numpy as np
import os

# ------------------------------------------------------------
# 1. パラメータ設定
# ------------------------------------------------------------
# psp.fac_matrix_make が作成した回転行列のtplot変数名
fac_matrix_tvar = 'tha_fgl_gsm_fac_mat'

# ------------------------------------------------------------
# 2. tplot変数からデータを取得し、角度を計算
# ------------------------------------------------------------
try:
    t_mat, d_mat = pt.get_data(fac_matrix_tvar)

    if t_mat is not None:
        # d_mat の形状が (N, 3, 3) であることを想定
        e3_z = d_mat[:, 2, 2]
        angle_deg = np.degrees(
            np.arccos(np.clip(e3_z, -1.0, 1.0))
        )
        
        # ------------------------------------------------------------
        # 3. 計算結果を新しいtplot変数として格納し、オプションを設定
        # ------------------------------------------------------------
        angle_tvar = 'B0_z_angle'
        pt.store_data(angle_tvar, data={'x': t_mat, 'y': angle_deg})
        
        # --- プロットのスタイルを設定 ---
        pt.options(angle_tvar, 'title', 'Rotation angle between B0 and GSM z-axis')
        pt.options(angle_tvar, 'ytitle', '∠(B0, z_gsm)')
        pt.options(angle_tvar, 'ysubtitle', '[deg]')
        pt.options(angle_tvar, 'grid', True)

        # ------------------------------------------------------------
        # 4. 条件に応じてプロットまたは画像保存
        # ------------------------------------------------------------
        if os.path.isdir(path_base_save_plot):
            # --- フォルダが存在する場合：表示せずに保存 ---
            save_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
            print(f"Plotting and saving to {save_path}...")
            
            pt.tplot(
                angle_tvar,
                save_png=save_path,
                display=False
            )
            plt.close('all')
            print("Done.")
            

        else:
            # --- フォルダが存在しない場合：通常表示 ---
            print("Displaying plot...")
            pt.tplot(angle_tvar)

except KeyError:
    print(f"tplot変数 '{fac_matrix_tvar}' が見つかりません。")
except Exception as e:
    print(f"エラーが発生しました: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 128                # サンプリング周波数 [Hz]
cutoff = 0.45           # カットオフ周波数 [Hz]  (spin=8 s → 0.125 Hz より少し高め)
order = 4               # 4次 Butterworth（2-pole × 2-stage）
spin_tone = 1/3         # spin周期 (ERG: 8 sec, THEMIS: 3 sec)

sos = butter(N=order, Wn=cutoff/(fs/2), btype='high', output='sos')

# ------------------ インパルス応答 ------------------
n = 2048                            # インパルス長さ（padlen より十分長く）
delta = np.zeros(n)
delta[n//2] = 1                    # 中央に δ(0)

h = sosfiltfilt(sos, delta)         # 0 相フィルタとして通す
t = (np.arange(n) - n//2) / fs      # 時間軸（0 を中心にシフト）

# ------------------ 周波数応答（片方向１回分） ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)   # |H(f)| 片方向

# filtfilt は 2 回かけるので、実際の振幅応答は |H(f)|^2 になる
H_dbl = np.abs(H)**2

# ------------------ プロット ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse response (zero-phase, 8-pole equivalent)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)| (double-pass)')
axs[1].axvline(cutoff, color='grey', ls='--', label=f'cutoff = {cutoff:.3f} Hz')
axs[1].axvline(spin_tone, color='green', ls='--', label=f'spin tone = {spin_tone:.3f} Hz')
axs[1].set_title('Magnitude response (double-pass, filtfilt)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20)
axs[1].set_xlim(4E-2)
#axs[1].set_yscale('symlog', linthresh=1E-1)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
# ------------------------------------------------------------
# 0. Butterworth HPF の係数（質問文と同じ）
# ------------------------------------------------------------
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt

fs, cutoff = 128.0, 0.45
sos = butter(4, cutoff/(fs/2), btype='high', output='sos')

def hp_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    padlen = 3 * (sos_mat.shape[0]-1)
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

# ------------------------------------------------------------
# 1. セグメント ID を列挙
# ------------------------------------------------------------
seg_ids = sorted({v.split('_seg')[-1] for v in pt.tnames('Ex_fac_seg*')})

# overplot まとめ変数用
Ex_hp, Ey_hp, Ez_hp = [], [], []
Bx_hp, By_hp, Bz_hp = [], [], []

# ------------------------------------------------------------
# 2. 各 seg で HPF → TVar 登録
# ------------------------------------------------------------
for sid in seg_ids:
    for base in ['Ex_fac', 'Ey_fac', 'Ez_fac', 'Bx_fac', 'By_fac', 'Bz_fac']:
        src = f'{base}_seg{sid}'
        da  = pt.data_quants[src]
        hp  = hp_segmented(da.values, sos)

        dst = f'{base}_hp_seg{sid}'
        pt.store_data(dst, data={'x': da.time.values, 'y': hp})

        # 保存しておいて後で overplot 用リストへ
        if   base == 'Ex_fac': Ex_hp.append(dst)
        elif base == 'Ey_fac': Ey_hp.append(dst)
        elif base == 'Ez_fac': Ez_hp.append(dst)
        elif base == 'Bx_fac': Bx_hp.append(dst)
        elif base == 'By_fac': By_hp.append(dst)
        elif base == 'Bz_fac': Bz_hp.append(dst)

        unit = '[mV/m]' if base.startswith('E') else '[nT]'
        pt.options(dst, 'ytitle', base.replace('_fac',' (FAC, HP)'))
        pt.options(dst, 'ysubtitle', unit)

# ------------------------------------------------------------
# 3. まとめ変数を作って overplot
# ------------------------------------------------------------
pt.store_data('Ex_fac_hp_all', data=Ex_hp)
pt.store_data('Ey_fac_hp_all', data=Ey_hp)
pt.store_data('Ez_fac_hp_all', data=Ez_hp)
pt.store_data('Bx_fac_hp_all', data=Bx_hp)
pt.store_data('By_fac_hp_all', data=By_hp)
pt.store_data('Bz_fac_hp_all', data=Bz_hp)

# ------------------------------------------------------------
# 4. 描画（6 パネル）
# ------------------------------------------------------------
vars_hp = ['Ex_fac_hp_all', 'Ey_fac_hp_all', 'Ez_fac_hp_all',
          'Bx_fac_hp_all', 'By_fac_hp_all', 'Bz_fac_hp_all']

if os.path.isdir(path_base_save_plot):
    # 存在する → 表示せずに保存のみ
    fig, axes = pt.tplot(
        vars_hp,
        display=False,
        return_plot_objects=True,
        save_png=os.path.join(path_base_save_plot, 'EB_fields_fac_hp.png')
    )
    plt.close(fig)
else:
    # 存在しない → 通常表示
    pt.tplot(
        vars_hp,
        display=True
    )


In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt

sys.path.append("..")
import module_handmade.tdwavelet as tw
importlib.reload(tw)

# ── 1. 各セグメントに対してウェーブレット計算とオプション設定 ───────────

# 各成分のtplot変数名を保存するためのリストを初期化
Ex_cwt, Ey_cwt, Ez_cwt = [], [], []
Bx_cwt, By_cwt, Bz_cwt = [], [], []

for sid in seg_ids:
    for EB, unit in [('E', '(mV/m)^2/Hz'), ('B', 'nT^2/Hz')]:
        for comp in ['x', 'y', 'z']:
            # セグメントIDを含む入力変数名を作成
            var_in = f'{EB}{comp}_fac_hp_seg{sid}'
            if var_in not in pt.data_quants:
                continue

            # ウェーブレット解析を実行
            # dt(サンプリング周期)はデータに合わせて要調整。ここでは仮に1/128秒としている。
            tw.tdwavelet(
                var_in,
                dt=1/128,
                s0=1/128*8E0,
                suffix='_cwt',
                zrange=[1e-6, 1e2]
            )

            # 出力変数名
            old_name = f'{var_in}_cwt'
            var_out = f'{EB}{comp}_fac_hp_cwt_seg{sid}'
            pt.tplot_rename(old_name, var_out)

            # 軸ラベルなどの設定
            pt.options(var_out, 'ylog', 1)
            pt.options(var_out, 'zlog', 1)
            pt.options(var_out, 'ztitle', 'PSD')
            pt.options(var_out, 'zsubtitle', unit)
            pt.options(var_out, 'ytitle', f'{EB}_{comp} (FAC)')
            pt.options(var_out, 'ysubtitle', '[Hz]')
            pt.options(var_out, 'colormap', 'turbo')
            pt.options(var_out, 'zrange', [1e-6, 1e2]) # z軸の範囲を統一
            pt.options(var_out, 'yrange', [0.45, 1E2])

            # 成分ごとにリストへ追加
            if   EB == 'E' and comp == 'x': Ex_cwt.append(var_out)
            elif EB == 'E' and comp == 'y': Ey_cwt.append(var_out)
            elif EB == 'E' and comp == 'z': Ez_cwt.append(var_out)
            elif EB == 'B' and comp == 'x': Bx_cwt.append(var_out)
            elif EB == 'B' and comp == 'y': By_cwt.append(var_out)
            elif EB == 'B' and comp == 'z': Bz_cwt.append(var_out)

# ── 2. セグメントを結合したリンク変数を作成 ─────────────────
# store_dataを使い、セグメント化されたスペクトルを一つにまとめる
pt.store_data('Ex_cwt_all', data=Ex_cwt)
pt.store_data('Ey_cwt_all', data=Ey_cwt)
pt.store_data('Ez_cwt_all', data=Ez_cwt)
pt.store_data('Bx_cwt_all', data=Bx_cwt)
pt.store_data('By_cwt_all', data=By_cwt)
pt.store_data('Bz_cwt_all', data=Bz_cwt)

In [ ]:
#import sys
#sys.path.append("..")
#import module_handmade.psd_plotter_themis as pp
#importlib.reload(pp)
#
## ── 3. タイムスパンを変えつつプロット／保存 ─────────────────
#time_windows = [
#    np.datetime64('2022-09-01T22:25'), np.datetime64('2022-09-01T22:30'), np.datetime64('2022-09-01T22:45'), np.datetime64('2022-09-01T22:50'), np.datetime64('2022-09-01T22:55'), np.datetime64('2022-09-01T23:00'), np.datetime64('2022-09-01T23:05')
#]
#
## プロット対象を「結合後」の変数リストに変更
#vars_to_plot = [
#    'Ex_fac_hp_cwt', 'Ey_fac_hp_cwt', 'Ez_fac_hp_cwt',
#    'Bx_fac_hp_cwt', 'By_fac_hp_cwt', 'Bz_fac_hp_cwt'
#]
#
#vars_to_plot_all_c = [
#    'Ex_fac_hp_cwt_all_c', 'Ey_fac_hp_cwt_all_c', 'Ez_fac_hp_cwt_all_c',
#    'Bx_fac_hp_cwt_all_c', 'By_fac_hp_cwt_all_c', 'Bz_fac_hp_cwt_all_c'
#]
#
#for var in vars_to_plot:
#    data = pp.concat_tplot_segments(var, seg_ids)
#    pt.store_data(f'{var}_all_c', data={'x': data.time, 'y': data.data, 'v': data.spec_bins})
#
#for t0 in time_windows:
#    start_str = str(t0)
#    pt.timespan(start_str, 5, keyword='minute')
#    print(f"Processing window: {start_str}")
#    
#    # オプション設定
#    for EB, unit in [('B', 'nT^2/Hz'), ('E', '(mV/m)^2/Hz')]:
#        for comp in ['x', 'y', 'z']:
#            var_in = f'{EB}{comp}_fac_hp_cwt_all_c'
#            if var_in not in pt.data_quants:
#                print(f'skip (no {var_in})')
#                continue
#            pt.options(var_in, 'spec', True)
#            pt.options(var_in, 'ylog', 1)
#            pt.options(var_in, 'zlog', 1)
#            pt.options(var_in, 'ztitle', 'PSD')
#            pt.options(var_in, 'zsubtitle', unit)
#            pt.options(var_in, 'ytitle', f'{EB}_{comp} (FAC)')
#            pt.options(var_in, 'ysubtitle', '[Hz]')
#            pt.options(var_in, 'colormap', 'turbo')
#            pt.options(var_in, 'zrange', [1e-6, 1e3])
#            pt.options(var_in, 'yrange', [0.45, 1E2])
#
#    # プロット・保存処理
#    if os.path.isdir(path_base_save_plot):
#        os.makedirs(path_base_save_plot, exist_ok=True)
#        fn_time = start_str.replace(':', '').replace('T', '_')
#        save_png = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        
#        print(f"  -> Plotting and saving...")
#        fig, axes = pt.tplot(
#            vars_to_plot_all_c,
#            display=False,
#            return_plot_objects=True,
#            save_png=save_png
#        )
#        plt.close(fig)
#    else:
#        print(f"  -> Plotting...")
#        pt.tplot(vars_to_plot_all_c, display=True)

# 軌道データから、衛星データ(GSM)を導出

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np

# 1. THEMIS-Aの軌道データをロード
psp.themis.state(
    probe='a',
    trange=time_range
)

vel_gsm   = pt.data_quants['tha_vel_gsm'].data   # (Nt,3)[R_E]
time_gsm = pt.data_quants['tha_vel_gsm'].time.values


time_gsm_s = time_gsm.astype('datetime64[ns]').astype(float) * 1e-9

pt.store_data('v_sc_gsm', data={'x': time_gsm_s, 'y': vel_gsm})
pt.options('v_sc_gsm', 'ytitle', r'$V_{\mathrm{sc}}$ (GSM) [km/s]')
pt.options('v_sc_gsm', 'legend_names', ['x','y','z'])

pt.timespan('2022-09-01/22:20:00', 1, keyword='hour')

if os.path.isdir(path_base_save_plot):
    png_path = os.path.join(path_base_save_plot, f'v_sc_gsm.png')
    pt.tplot('v_sc_gsm', save_png=png_path, display=False)
    plt.close('all')
    print(f"Saved: {png_path}")
else:
    pt.tplot('v_sc_gsm', display=True)

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp # pyspedasもインポートしておく
import matplotlib.pyplot as plt
import os

# --- 0. パラメータ設定 ---
# 宇宙機速度と回転行列のtplot変数名
v_sc_gsm_tvar = 'v_sc_gsm' # この変数は事前にロードされている必要がある
fac_matrix_tvar = 'tha_fgl_gsm_fac_mat' # FAC変換スクリプトで作成済

# --- 1. 回転行列データを準備 ---
try:
    # xarray=Trueでオブジェクト全体を取得。これにより時間軸がdatetime64になる
    matrix_da_raw = pt.get_data(fac_matrix_tvar, xarray=True)
    
    if matrix_da_raw is None:
        raise KeyError(f"tplot変数 '{fac_matrix_tvar}' のデータ取得に失敗しました。")
    
    # datetime64型の時間軸と、値(numpy配列)を取得
    t_mat = matrix_da_raw.time.values
    d_mat = matrix_da_raw.values

    # 以前の議論の通り、行列が2Dの場合に3Dに修正
    if d_mat.ndim == 2:
        d_mat = d_mat[np.newaxis, :, :]

    # datetime64型のt_matを使って、補間用のDataArrayを準備
    matrix_da = xr.DataArray(d_mat, dims=('time_mat', 'row', 'col'), coords={'time_mat': t_mat})
    print(f"回転行列'{fac_matrix_tvar}'を準備しました。Shape: {matrix_da.shape}")

except KeyError:
    print(f"エラー: 回転行列'{fac_matrix_tvar}'が見つかりません。FAC変換スクリプトを先に実行してください。")
    exit()

# --- 2. 各セグメントをループして処理 ---
V_sc_fac_vars = []
V_sc_fac_perp_vars = []

for sid in seg_ids:
    # 各セグメントの基準となる時間軸を取得
    # （この変数は時間軸の参照にしか使わない）
    try:
        # .times でfloat配列を取得する代わりに、xarray=Trueでオブジェクト全体を取得
        psd_da = pt.get_data(f'Bx_fac_hp_cwt_seg{sid}', xarray=True)
        # .time でdatetime64型を持つtime座標を取得する
        t_psd_datetime = psd_da.time
    except (KeyError, AttributeError):
        print(f"セグメント{sid}の時間軸が見つかりません。スキップします。")
        continue

    # 元の宇宙機速度データをセグメントの時間軸に補間
    V_sc_gsm_interp_da = pt.get_data(v_sc_gsm_tvar, xarray=True).interp(time=t_psd_datetime, method='linear')

    R_interp_np = matrix_da.interp(time_mat=t_psd_datetime, method="linear").values
    V_sc_fac = np.einsum('tij,tj->ti', R_interp_np, V_sc_gsm_interp_da.values)

    V_sc_fac_perp = np.sqrt(V_sc_fac[:, 0]**2 + V_sc_fac[:, 1]**2)

    # --- tplotへの格納とオプション設定 ---
    new_name = f'v_sc_fac_seg{sid}'
    new_name_perp = f'v_sc_fac_perp_seg{sid}'
    V_sc_fac_vars.append(new_name)
    V_sc_fac_perp_vars.append(new_name_perp)

    pt.store_data(new_name, data={'x': t_psd_datetime, 'y': V_sc_fac})
    pt.store_data(new_name_perp, data={'x': t_psd_datetime, 'y': V_sc_fac_perp})

    
    pt.options(new_name, 'ytitle', r'$V_{\mathrm{sc}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name, 'legend_names', None)
    pt.options(new_name, 'ysubtitle', r'[km/s]')
    pt.options(new_name_perp, 'ytitle', r'$V_{\mathrm{sc}\perp}$ (FAC)')
    pt.options(new_name_perp, 'ysubtitle', r'[km/s]')

# --- 3. まとめ変数とプロット (元のコードと同じ) ---
pt.store_data('v_sc_fac_all', data=V_sc_fac_vars)
pt.store_data('v_sc_fac_perp_all', data=V_sc_fac_perp_vars)
vars_to_plot = ['v_sc_fac_all', 'v_sc_fac_perp_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'v_sc_fac_and_perp.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.mom(trange=time_range, probe='a', level='l2')

In [ ]:
pt.tplot(['tha_peim_data_quality', 'tha_peem_data_quality'])
print(pt.data_quants['tha_peim_data_quality'].attrs)
print(pt.data_quants['tha_peem_data_quality'].attrs)

In [ ]:
import xarray as xr
import pytplot as pt
import numpy as np

ND_electron     = pt.data_quants['tha_peem_density']                # [/cc]
Temp_electron   = pt.data_quants['tha_peem_ptot'] / ND_electron     # [eV]

pt.store_data('Temp_electron', data={'x': Temp_electron.time, 'y': Temp_electron.data})

ND_ion          = pt.data_quants['tha_peim_density']                # [/cc]
Temp_ion        = pt.data_quants['tha_peim_ptot'] / ND_ion          # [eV]
Velocity_ion_gsm= pt.data_quants['tha_peim_velocity_gsm']           # [km/s]

pt.store_data('Temp_ion', data={'x': Temp_ion.time, 'y': Temp_ion.data})

print(ND_electron)
print(pt.data_quants['tha_peem_ptot'])
print(Temp_electron)
print(ND_ion)
print(Temp_ion)
print(Velocity_ion_gsm)

pt.options('tha_peem_density', 'ytitle', r'$n_{\mathrm{e}}$ [/cc]')
pt.options('Temp_electron', 'ytitle', r'$T_{\mathrm{e}}$ [eV]')

pt.options('tha_peim_density', 'ytitle', r'$n_{\mathrm{i}}$ [/cc]')
pt.options('Temp_ion', 'ytitle', r'$T_{\mathrm{i}}$ [eV]')
pt.options('tha_peim_velocity_gsm', 'ytitle', r'$V_{\mathrm{i}}$ (gsm) [km/s]')

vars_to_plot = ['tha_peem_density']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_density.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

vars_to_plot = ['tha_peem_density', 'Temp_electron']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_parameter.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

vars_to_plot = ['tha_peim_density', 'Temp_ion', 'tha_peim_velocity_gsm']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'ion_parameter.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp
import matplotlib.pyplot as plt
import os
import pandas as pd

# --- 0. パラメータ設定 ---
v_ion_gsm_tvar = 'tha_peim_velocity_gsm'
v_sc_gsm_tvar = 'v_sc_gsm'
fac_matrix_tvar = 'tha_fgl_gsm_fac_mat'

# --- 1. 回転行列データを準備 ---
try:
    matrix_da_raw = pt.get_data(fac_matrix_tvar, xarray=True)
    if matrix_da_raw is None: raise KeyError
    t_mat = matrix_da_raw.time.values
    d_mat = matrix_da_raw.values
    if d_mat.ndim == 2: d_mat = d_mat[np.newaxis, :, :]
    matrix_da = xr.DataArray(d_mat, dims=('time_mat', 'row', 'col'), coords={'time_mat': t_mat})
    print(f"回転行列'{fac_matrix_tvar}'を準備しました。")
except KeyError:
    print(f"エラー: 回転行列'{fac_matrix_tvar}'が見つかりません。")
    exit()

# --- 2. 速度データを準備 ---
V_ion_gsm = pt.get_data(v_ion_gsm_tvar, xarray=True)
V_sc_gsm = pt.get_data(v_sc_gsm_tvar, xarray=True)

# --- 3. 各セグメントをループして処理 ---
v_ion_fac_vars, v_ion_fac_perp_vars = [], []
v_sys_fac_vars, v_sys_fac_perp_vars, v_sys_fac_perp_rolling_vars = [], [], []

for sid in seg_ids:
    try:
        t_psd = pt.get_data(f'Bx_fac_hp_cwt_seg{sid}', xarray=True).time
    except (KeyError, AttributeError):
        print(f"セグメント{sid}の時間軸が見つかりません。スキップします。")
        continue

    # --- 物理量をセグメントの時間軸に補間 ---
    V_ion_gsm_interp = V_ion_gsm.interp(time=t_psd, method='linear').values
    V_sc_gsm_interp = V_sc_gsm.interp(time=t_psd, method='linear').values
    R_interp = matrix_da.interp(time_mat=t_psd, method="linear").values

    # --- NumPyでFACへ変換 ---
    V_ion_fac = np.einsum('tij,tj->ti', R_interp, V_ion_gsm_interp)
    V_sc_fac = np.einsum('tij,tj->ti', R_interp, V_sc_gsm_interp)

    # --- 物理量の計算 ---
    V_sys_fac = V_ion_fac - V_sc_fac
    V_ion_fac_perp = np.sqrt(V_ion_fac[:, 0]**2 + V_ion_fac[:, 1]**2)
    V_sys_fac_perp = np.sqrt(V_sys_fac[:, 0]**2 + V_sys_fac[:, 1]**2)
    
    # --- tplotへの格納とオプション設定 ---
    # (イオン速度)
    new_name = f'v_ion_fac_seg{sid}'
    new_name_perp = f'v_ion_fac_perp_seg{sid}'
    v_ion_fac_vars.append(new_name)
    v_ion_fac_perp_vars.append(new_name_perp)

    pt.store_data(new_name, data={'x': t_psd, 'y': V_ion_fac})
    pt.store_data(new_name_perp, data={'x': t_psd, 'y': V_ion_fac_perp})

    pt.options(new_name, 'ytitle', r'$V_{\mathrm{i}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name, 'legend_names', None)
    pt.options(new_name, 'ysubtitle', r'[km/s]')
    pt.options(new_name, 'char_size', 15)

    pt.options(new_name_perp, 'ytitle', r'$V_{\mathrm{i}\perp}$ (FAC)')
    pt.options(new_name_perp, 'char_size', 15)
    pt.options(new_name_perp, 'ysubtitle', r'[km/s]')

    # (システム速度)
    new_name_sys = f'v_sys_fac_seg{sid}'
    new_name_sys_perp = f'v_sys_fac_perp_seg{sid}'
    v_sys_fac_vars.append(new_name_sys)
    v_sys_fac_perp_vars.append(new_name_sys_perp)

    pt.store_data(new_name_sys, data={'x': t_psd, 'y': V_sys_fac})
    pt.store_data(new_name_sys_perp, data={'x': t_psd, 'y': V_sys_fac_perp})

    pt.options(new_name_sys, 'ytitle', r'$V_{\mathrm{sys}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name_sys, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name_sys, 'legend_names', None)
    pt.options(new_name_sys, 'ysubtitle', r'[km/s]')
    pt.options(new_name_sys, 'char_size', 15)

    pt.options(new_name_sys_perp, 'ytitle', r'$V_{\mathrm{sys}\perp}$ (FAC)')
    pt.options(new_name_sys_perp, 'char_size', 15)
    pt.options(new_name_sys_perp, 'ysubtitle', r'[km/s]')
    
    # (移動平均)
    temp_da = xr.DataArray(V_sys_fac_perp, dims=('time',), coords={'time': t_psd})
    dt_i = np.nanmedian(np.diff(t_psd.values)).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    V_sys_fac_perp_rolling = temp_da.rolling(time=win_pts_i, center=True).mean()
    
    new_name_sys_perp_rolling = f'v_sys_fac_perp_rolling_seg{sid}'
    v_sys_fac_perp_rolling_vars.append(new_name_sys_perp_rolling)
    pt.store_data(new_name_sys_perp_rolling, data={'x': t_psd, 'y': V_sys_fac_perp_rolling.data})

# --- 4. まとめ変数とプロット ---
pt.store_data('v_ion_fac_all', data=v_ion_fac_vars)
pt.store_data('v_ion_fac_perp_all', data=v_ion_fac_perp_vars)
vars_to_plot_1 = ['v_ion_fac_all', 'v_ion_fac_perp_all']

pt.store_data('v_sys_fac_all', data=v_sys_fac_vars)
pt.store_data('v_sys_fac_perp_all', data=v_sys_fac_perp_vars)
vars_to_plot_2 = ['v_sys_fac_all', 'v_sys_fac_perp_all']

pt.store_data('v_sys_fac_perp_rolling_all', data=v_sys_fac_perp_rolling_vars)
vars_to_plot_3 = ['v_sys_fac_perp_rolling_all']

# プロットを3回に分けて実行
if os.path.isdir(path_base_save_plot):
    save_png_1 = os.path.join(path_base_save_plot, 'v_ion_fac_and_perp.png')
    pt.tplot(vars_to_plot_1, display=False, save_png=save_png_1)
    print(f"Saved plot to {save_png_1}")

    save_png_2 = os.path.join(path_base_save_plot, 'v_sys_fac_and_perp.png')
    pt.tplot(vars_to_plot_2, display=False, save_png=save_png_2)
    print(f"Saved plot to {save_png_2}")
    
    save_png_3 = os.path.join(path_base_save_plot, 'v_sys_fac_perp_rolling.png')
    pt.tplot(vars_to_plot_3, display=False, save_png=save_png_3)
    print(f"Saved plot to {save_png_3}")
    
    plt.close('all')
else:
    print("--- V_ion plots ---")
    pt.tplot(vars_to_plot_1)
    print("\n--- V_sys plots ---")
    pt.tplot(vars_to_plot_2)
    print("\n--- V_sys_perp_rolling plots ---")
    pt.tplot(vars_to_plot_3)

In [ ]:
import xarray as xr
import pytplot as pt
import numpy as np

ion_acoustic_speed_vars, ion_thermal_speed_vars, Alfven_speed_vars, ion_cyclo_freq_vars, beta_ion_vars, tau_vars = [], [], [], [], [], []

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

for sid in seg_ids:
    psd_var = f'Bx_fac_hp_cwt_seg{sid}'

    psd_da = pt.data_quants[psd_var]
    t_psd = psd_da.time

    ion_acoustic_speed = np.sqrt(pt.data_quants['Temp_electron'] * elementary_charge / proton_mass).interp(time=t_psd, method='linear')
    dt_i = (ion_acoustic_speed.time[1] - ion_acoustic_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_acoustic_speed = ion_acoustic_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_acoustic = f'ion_acoustic_speed_seg{sid}'
    ion_acoustic_speed_vars.append(new_ion_acoustic)

    pt.store_data(new_ion_acoustic, data={'x': t_psd, 'y': ion_acoustic_speed*1E-3})
    pt.options(new_ion_acoustic, 'ytitle', r'$c_{\mathrm{s}}$')
    pt.options(new_ion_acoustic, 'ysubtitle', '[km/s]')


    ion_thermal_speed = np.sqrt(2E0 * pt.data_quants['Temp_ion'] * elementary_charge / proton_mass).interp(time=t_psd, method='linear')
    dt_i = (ion_thermal_speed.time[1] - ion_thermal_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_thermal_speed = ion_thermal_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_thermal = f'ion_thermal_speed_seg{sid}'
    ion_thermal_speed_vars.append(new_ion_thermal)

    pt.store_data(new_ion_thermal, data={'x': t_psd, 'y': ion_thermal_speed*1E-3})
    pt.options(new_ion_thermal, 'ytitle', r'$v_{\mathrm{thi}}$')
    pt.options(new_ion_thermal, 'ysubtitle', '[km/s]')


    B_total = pt.data_quants['tha_fgh_btotal'].interp(time=t_psd, method='linear')
    ND_electron_interp = ND_electron.interp(time=t_psd, method='linear')
    Alfven_speed = B_total*1E-9 / np.sqrt(mu0 * proton_mass * ND_electron_interp*1E6)
    dt_i = (Alfven_speed.time[1] - Alfven_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    Alfven_speed = Alfven_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_Alfven_speed = f'Alfven_speed_seg{sid}'
    Alfven_speed_vars.append(new_Alfven_speed)

    pt.store_data(new_Alfven_speed, data={'x': t_psd, 'y': Alfven_speed*1E-3})
    pt.options(new_Alfven_speed, 'ytitle', r'$v_{\mathrm{A}}$')
    pt.options(new_Alfven_speed, 'ysubtitle', '[km/s]')


    ion_cyclo_freq = elementary_charge * B_total*1E-9 / proton_mass / 2E0 / np.pi
    dt_i = (ion_cyclo_freq.time[1] - ion_cyclo_freq.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_cyclo_freq = ion_cyclo_freq.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_cyclo_freq = f'ion_cyclo_freq_seg{sid}'
    ion_cyclo_freq_vars.append(new_ion_cyclo_freq)

    pt.store_data(new_ion_cyclo_freq, data={'x': t_psd, 'y': ion_cyclo_freq})
    pt.options(new_ion_cyclo_freq, 'ytitle', r'$f_{\mathrm{ci}}$')
    pt.options(new_ion_cyclo_freq, 'ysubtitle', '[Hz]')

    
    beta_ion = (ion_thermal_speed / Alfven_speed)**2E0
    dt_i = (beta_ion.time[1] - beta_ion.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    beta_ion = beta_ion.rolling(time=win_pts_i, center=True).mean('time')

    new_beta_ion = f'beta_ion_seg{sid}'
    beta_ion_vars.append(new_beta_ion)

    pt.store_data(new_beta_ion, data={'x': t_psd, 'y': beta_ion})
    pt.options(new_beta_ion, 'ytitle', r'$\beta_{\mathrm{i}}$')
    
    tau = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0
    dt_i = (tau.time[1] - tau.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    tau = tau.rolling(time=win_pts_i, center=True).mean('time')

    new_tau = f'tau_seg{sid}'
    tau_vars.append(new_tau)

    pt.store_data(new_tau, data={'x': t_psd, 'y': tau})
    pt.options(new_tau, 'ytitle', r'$\tau$')

pt.store_data('ion_thermal_speed_all', data=ion_thermal_speed_vars)
pt.store_data('ion_acoustic_speed_all', data=ion_acoustic_speed_vars)
pt.store_data('Alfven_speed_all', data=Alfven_speed_vars)
pt.store_data('ion_cycl_freq_all', data=ion_cyclo_freq_vars)

vars_to_plot = ['ion_thermal_speed_all', 'ion_acoustic_speed_all', 'Alfven_speed_all', 'ion_cycl_freq_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_1.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )


pt.store_data('beta_ion_all', data=beta_ion_vars)
pt.store_data('tau_all', data=tau_vars)

vars_to_plot = ['beta_ion_all', 'tau_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_2.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_1sec_avg'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（モジュール関数を呼び出す）
# ------------------------------------------------------------
data_dict = pp.load_and_prepare_data(cutoff_freq=0.45)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 解析対象の時間を決め、1秒ごとにループ
# ------------------------------------------------------------

def process_and_save_plot(t_start, data_dict, out_dir):
    """
    指定された単一の時刻について、スペクトルをプロットし、画像を保存する関数。
    """
    # モジュール関数を呼び出してプロットを作成
    # psd_plotter を pp としてインポートしている前提
    fig = pp.plot_freq_spectrum(t_start, data_dict, interval_sec=1, n_samples_mc=200)
    
    # figがNoneでなければ（データがあってプロットが作成されれば）保存
    if fig is not None:
        try:
            # ファイル名に使いやすいように文字列に変換
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=200)
        finally:
            # 保存に失敗しても、メモリ解放のために必ずクローズする
            plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)

#process_and_save_plot(t_min, data_dict, out_dir)

# pandas.date_rangeで1秒ごとのタイムスタンプを生成
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_plot)(t_start, data_dict, out_dir) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_1sec_avg_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=0.45)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    fig = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=1, 
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range
    )
    
    if fig is not None:
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig) # メモリ解放
            gc.collect()   # ガベージコレクション

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:28:00', '2022-09-01T23:10:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e3)
n_bins_to_use       = 30
fit_range_to_use    = (3, 300)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use

    ) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_30sec_avg_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=0.45)

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    fig = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=1, 
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range
    )
    
    if fig is not None:
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig) # メモリ解放
            gc.collect()   # ガベージコレクション

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:28:00', '2022-09-01T23:10:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='30s')

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e3)
n_bins_to_use       = 50
fit_range_to_use    = (3, 300)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use

    ) for t_start in time_steps
)

print('Finished saving all plots!')